In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TEST_START = pd.Timestamp("2022-09-08")

print("Project root:", PROJECT_ROOT)
print("Processed directory exists:", PROCESSED_DIR.is_dir())


Project root: c:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection
Processed directory exists: True


In [2]:
if not PROCESSED_DIR.is_dir():
    raise FileNotFoundError(f"Processed data directory not found: {PROCESSED_DIR}")

candidates = sorted(PROCESSED_DIR.rglob("*.parquet"))
for p in candidates:
    print(f"{p.relative_to(PROJECT_ROOT)} | {p.stat().st_size / 1024**2:.1f} MB")

print(f"\nFound {len(candidates)} parquet files.")


data\processed\test.parquet | 59.1 MB
data\processed\testing_data.parquet | 80.0 MB
data\processed\train.parquet | 166.6 MB
data\processed\training_data.parquet | 224.3 MB
data\processed\v3e1_dev_sep1_7.parquet | 326.3 MB
data\processed\val.parquet | 39.5 MB

Found 6 parquet files.


In [3]:
DEV_SOURCE_PATH = PROCESSED_DIR / "training_data.parquet"
TEST_SOURCE_PATH = PROCESSED_DIR / "testing_data.parquet"

assert DEV_SOURCE_PATH.is_file(), f"Missing development source: {DEV_SOURCE_PATH}"
if TEST_SOURCE_PATH is None:
    print(
        "Set TEST_SOURCE_PATH to the verified pre-V3-E1 holdout parquet, then rerun this cell."
    )
else:
    TEST_SOURCE_PATH = Path(TEST_SOURCE_PATH)
    assert TEST_SOURCE_PATH.is_file(), f"Missing holdout source: {TEST_SOURCE_PATH}"
    assert TEST_SOURCE_PATH.resolve() != DEV_SOURCE_PATH.resolve(), (
        "Test and development paths are identical"
    )
    print("Development source:", DEV_SOURCE_PATH)
    print("Holdout source:", TEST_SOURCE_PATH)


Development source: c:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\processed\training_data.parquet
Holdout source: c:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\processed\testing_data.parquet


# Validate columns and time boundaries

In [4]:
if TEST_SOURCE_PATH is None:
    raise ValueError("Set TEST_SOURCE_PATH in the previous cell before validation.")

required = {
    "transaction_timestamp",
    "from_bank_id",
    "from_account_id",
    "to_bank_id",
    "to_account_id",
    "amount_paid",
    "amount_received",
    "payment_currency",
    "receiving_currency",
    "payment_format",
}


In [5]:
import pyarrow.parquet as pq

for label, path in [("Development", DEV_SOURCE_PATH), ("Holdout", TEST_SOURCE_PATH)]:
    pfile = pq.ParquetFile(path)
    names = set(pfile.schema_arrow.names)
    missing = sorted(required - names)
    print(f"{label}: {pfile.metadata.num_rows:,} rows")
    print(f"  missing required columns: {missing}")
    if missing:
        raise ValueError(f"{label} missing required source fields: {missing}")


Development: 3,731,672 rows
  missing required columns: []
Holdout: 1,346,673 rows
  missing required columns: []


In [6]:
# Read just timestamps to verify chronology.
dev_times = pd.to_datetime(
    pd.read_parquet(DEV_SOURCE_PATH, columns=["transaction_timestamp"])[
        "transaction_timestamp"
    ]
)
test_times = pd.to_datetime(
    pd.read_parquet(TEST_SOURCE_PATH, columns=["transaction_timestamp"])[
        "transaction_timestamp"
    ]
)

In [7]:
assert len(dev_times) == 3_731_672, (
    "Development source count differs from the final model's training rows"
)
assert dev_times.max() < TEST_START, "Development contains September 8+ timestamps"
assert test_times.min() >= TEST_START, "Holdout contains development-period timestamps"
assert not test_times.isna().any(), "Missing test timestamps"

print(f"Development: {len(dev_times):,} rows, {dev_times.min()} to {dev_times.max()}")
print(f"Holdout: {len(test_times):,} rows, {test_times.min()} to {test_times.max()}")
print(
    "Source schema and chronological boundaries verified. No labels or model predictions accessed."
)


Development: 3,731,672 rows, 2022-09-01 00:00:00 to 2022-09-07 23:59:00
Holdout: 1,346,673 rows, 2022-09-08 00:00:00 to 2022-09-18 16:18:00
Source schema and chronological boundaries verified. No labels or model predictions accessed.


# Combine development and holdout transaction history

In [8]:
import numpy as np
import pandas as pd

dev_source = pd.read_parquet(DEV_SOURCE_PATH)
test_source = pd.read_parquet(TEST_SOURCE_PATH)

TEST_START = pd.Timestamp("2022-09-08")


In [9]:
for df in (dev_source, test_source):
    df["transaction_timestamp"] = pd.to_datetime(df["transaction_timestamp"])

In [10]:
assert (dev_source["transaction_timestamp"] < TEST_START).all()
assert (test_source["transaction_timestamp"] >= TEST_START).all()


In [11]:
history_columns = [col for col in dev_source.columns if col != "is_laundering"]

assert set(history_columns).issubset(test_source.columns)

In [12]:
# Preserve original row identities
dev_source["_source"] = "development"
test_source["_source"] = "holdout"

dev_source["_original_row"] = np.arange(len(dev_source))
test_source["_original_row"] = np.arange(len(test_source))

combined_df = pd.concat(
    [
        dev_source.drop(columns=["is_laundering"]),
        test_source.drop(columns=["is_laundering"]),
    ],
    ignore_index=True,
)

combined_df = combined_df.sort_values(
    "transaction_timestamp",
    kind="stable",
).reset_index(drop=True)

print("Combined transactions:", f"{len(combined_df):,}")
print("Development rows:", f"{(combined_df['_source'] == 'development').sum():,}")
print("Holdout rows:", f"{(combined_df['_source'] == 'holdout').sum():,}")
print(
    "Date range:",
    combined_df["transaction_timestamp"].min(),
    "to",
    combined_df["transaction_timestamp"].max(),
)

print("Labels present:", "is_laundering" in combined_df.columns)
print("Duplicate transaction IDs:", combined_df["transaction_id"].duplicated().sum())


Combined transactions: 5,078,345
Development rows: 3,731,672
Holdout rows: 1,346,673
Date range: 2022-09-01 00:00:00 to 2022-09-18 16:18:00
Labels present: False
Duplicate transaction IDs: 0


# Recreate the first historical feature and verify development parity

In [13]:
def add_sender_tx_count_1h(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Count transactions made by the same sender during the
    previous 1 hour.

    Window:
        [t - 1 hour, t)

    Therefore:
    - current transaction is excluded
    - all transactions at the exact current timestamp are excluded
    - only strictly earlier activity is used

    Sender identity:
        from_bank_id + from_account_id
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "transaction_timestamp",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    sender_cols = [
        "from_bank_id",
        "from_account_id",
    ]

    # ---------------------------------------------------------
    # 1. Count how many transactions each sender made at each
    #    exact timestamp.
    # ---------------------------------------------------------

    timestamp_counts = (
        result.groupby(
            sender_cols + ["transaction_timestamp"],
            dropna=False,
        )
        .size()
        .rename("tx_at_timestamp")
        .reset_index()
    )

    timestamp_counts = timestamp_counts.sort_values(
        sender_cols + ["transaction_timestamp"]
    )

    # ---------------------------------------------------------
    # 2. For every sender/timestamp, sum transactions from
    #    the previous hour, excluding this exact timestamp.
    # ---------------------------------------------------------

    rolling_counts = (
        timestamp_counts.set_index("transaction_timestamp")
        .groupby(
            sender_cols,
            dropna=False,
        )["tx_at_timestamp"]
        .rolling(
            window="1h",
            closed="left",
        )
        .sum()
        .fillna(0)
        .rename("sender_tx_count_1h")
        .reset_index()
    )

    rolling_counts["sender_tx_count_1h"] = rolling_counts["sender_tx_count_1h"].astype(
        "int32"
    )

    # ---------------------------------------------------------
    # 3. Merge the historical count back onto every transaction.
    #
    # All transactions from the same sender at the same exact
    # timestamp receive the same past-history count.
    # ---------------------------------------------------------

    result = result.merge(
        rolling_counts[
            sender_cols
            + [
                "transaction_timestamp",
                "sender_tx_count_1h",
            ]
        ],
        on=sender_cols + ["transaction_timestamp"],
        how="left",
        validate="many_to_one",
        sort=False,
    )

    return result

In [14]:
# Compute the first historical feature on the combined chronological history.
# Do not use labels or model predictions.
assert "is_laundering" not in combined_df.columns
assert combined_df["transaction_id"].is_unique

v3a1_combined_df = add_sender_tx_count_1h(combined_df)

assert len(v3a1_combined_df) == len(combined_df)
assert v3a1_combined_df["transaction_id"].is_unique
assert v3a1_combined_df["sender_tx_count_1h"].notna().all()
assert (v3a1_combined_df["sender_tx_count_1h"] >= 0).all()

print("Feature computed:", "sender_tx_count_1h")
print("Rows:", f"{len(v3a1_combined_df):,}")
print("Null values:", int(v3a1_combined_df["sender_tx_count_1h"].isna().sum()))
print(
    "Min / max:",
    v3a1_combined_df["sender_tx_count_1h"].min(),
    "/",
    v3a1_combined_df["sender_tx_count_1h"].max(),
)


Feature computed: sender_tx_count_1h
Rows: 5,078,345
Null values: 0
Min / max: 0 / 2829


In [15]:
FROZEN_DEV_PATH = PROCESSED_DIR / "v3e1_dev_sep1_7.parquet"
assert FROZEN_DEV_PATH.is_file(), f"Missing frozen reference: {FROZEN_DEV_PATH}"

# Read only the two columns needed for the comparison.
frozen_first_feature = pd.read_parquet(
    FROZEN_DEV_PATH,
    columns=["transaction_id", "sender_tx_count_1h"],
)
assert frozen_first_feature["transaction_id"].is_unique


In [16]:
recomputed_dev = v3a1_combined_df.loc[
    v3a1_combined_df["_source"] == "development",
    ["transaction_id", "sender_tx_count_1h"],
]

parity = recomputed_dev.merge(
    frozen_first_feature,
    on="transaction_id",
    how="outer",
    suffixes=("_recomputed", "_frozen"),
    indicator=True,
    validate="one_to_one",
)

missing_rows = int((parity["_merge"] != "both").sum())
mismatch_rows = int(
    (
        parity["_merge"].eq("both")
        & parity["sender_tx_count_1h_recomputed"].ne(
            parity["sender_tx_count_1h_frozen"]
        )
    ).sum()
)

print("Development rows compared:", f"{len(recomputed_dev):,}")
print("Unmatched transaction IDs:", missing_rows)
print("Feature mismatches:", mismatch_rows)

assert missing_rows == 0, "Transaction IDs did not align. Stop and investigate."
assert mismatch_rows == 0, "Historical feature parity failed. Stop and investigate."
print("PASS: sender_tx_count_1h exactly matches frozen V3-E1 development values.")


Development rows compared: 3,731,672
Unmatched transaction IDs: 0
Feature mismatches: 0
PASS: sender_tx_count_1h exactly matches frozen V3-E1 development values.


# Same-currency 24-hour sender monetary volume

In [17]:
def add_sender_amount_sum_24h(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add the sender's total amount paid during the previous
    24 hours in the same payment currency as the current
    transaction.

    Historical window:
        [t - 24 hours, t)

    Therefore:
    - current transaction is excluded
    - transactions at the exact current timestamp are excluded
    - future transactions are excluded

    Sender identity:
        from_bank_id + from_account_id

    Currency:
        payment_currency
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "transaction_timestamp",
        "payment_currency",
        "amount_paid",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    # Temporary key allows missing currencies to be handled
    # consistently without changing the original column.
    result["_payment_currency_key"] = (
        result["payment_currency"].astype("string").fillna("__MISSING__")
    )

    sender_currency_cols = [
        "from_bank_id",
        "from_account_id",
        "_payment_currency_key",
    ]

    # ---------------------------------------------------------
    # 1. Aggregate amount paid by sender + currency
    #    at each exact timestamp.
    #
    # This is important because multiple transactions may
    # occur at exactly the same timestamp.
    # ---------------------------------------------------------

    timestamp_amounts = (
        result.groupby(
            sender_currency_cols + ["transaction_timestamp"],
            dropna=False,
            as_index=False,
        )["amount_paid"]
        .sum()
        .rename(columns={"amount_paid": "amount_paid_at_timestamp"})
    )

    timestamp_amounts = timestamp_amounts.sort_values(
        sender_currency_cols + ["transaction_timestamp"]
    )

    # ---------------------------------------------------------
    # 2. Sum only strictly earlier activity over 24 hours.
    # ---------------------------------------------------------

    rolling_amounts = (
        timestamp_amounts.set_index("transaction_timestamp")
        .groupby(
            sender_currency_cols,
            dropna=False,
        )["amount_paid_at_timestamp"]
        .rolling(
            window="24h",
            closed="left",
        )
        .sum()
        .fillna(0)
        .rename("sender_amount_paid_sum_24h_same_currency")
        .reset_index()
    )

    # ---------------------------------------------------------
    # 3. Merge historical value back to each transaction.
    # ---------------------------------------------------------

    result = result.merge(
        rolling_amounts[
            sender_currency_cols
            + [
                "transaction_timestamp",
                "sender_amount_paid_sum_24h_same_currency",
            ]
        ],
        on=(sender_currency_cols + ["transaction_timestamp"]),
        how="left",
        validate="many_to_one",
        sort=False,
    )

    # Heavy-tail-safe representation for the model.
    result["log_sender_amount_paid_sum_24h_same_currency"] = np.log1p(
        result["sender_amount_paid_sum_24h_same_currency"].clip(lower=0)
    )

    result = (
        result.sort_values("_row_order")
        .drop(
            columns=[
                "_row_order",
                "_payment_currency_key",
            ]
        )
        .reset_index(drop=True)
    )

    return result

In [18]:
assert "is_laundering" not in v3a1_combined_df.columns
v3a3_combined_df = add_sender_amount_sum_24h(v3a1_combined_df)
feature = "log_sender_amount_paid_sum_24h_same_currency"
raw_feature = "sender_amount_paid_sum_24h_same_currency"
assert len(v3a3_combined_df) == len(v3a1_combined_df)
assert v3a3_combined_df["transaction_id"].is_unique
assert v3a3_combined_df[feature].notna().all()
assert (v3a3_combined_df[raw_feature] >= 0).all()
print("Feature computed:", feature)
print("Rows:", f"{len(v3a3_combined_df):,}")
print("Null values:", int(v3a3_combined_df[feature].isna().sum()))
print(
    "Min / max:", v3a3_combined_df[feature].min(), "/", v3a3_combined_df[feature].max()
)


Feature computed: log_sender_amount_paid_sum_24h_same_currency
Rows: 5,078,345
Null values: 0
Min / max: 0.0 / 27.6438907735853


In [19]:
frozen_second = pd.read_parquet(
    FROZEN_DEV_PATH,
    columns=["transaction_id", "log_sender_amount_paid_sum_24h_same_currency"],
)
recomputed_second = v3a3_combined_df.loc[
    v3a3_combined_df["_source"].eq("development"),
    ["transaction_id", "log_sender_amount_paid_sum_24h_same_currency"],
]
assert frozen_second["transaction_id"].is_unique
parity2 = recomputed_second.merge(
    frozen_second,
    on="transaction_id",
    how="outer",
    suffixes=("_recomputed", "_frozen"),
    indicator=True,
    validate="one_to_one",
)
missing_rows = int(parity2["_merge"].ne("both").sum())
a = parity2["log_sender_amount_paid_sum_24h_same_currency_recomputed"].to_numpy()
b = parity2["log_sender_amount_paid_sum_24h_same_currency_frozen"].to_numpy()
close = np.isclose(a, b, rtol=1e-10, atol=1e-10, equal_nan=False)
mismatch_rows = int((~close).sum())
max_abs_difference = float(np.nanmax(np.abs(a - b)))
print("Development rows compared:", f"{len(recomputed_second):,}")
print("Unmatched transaction IDs:", missing_rows)
print("Feature mismatches (tolerance 1e-10):", mismatch_rows)
print("Maximum absolute difference:", max_abs_difference)
assert missing_rows == 0, "Transaction IDs do not align. Stop."
assert mismatch_rows == 0, "Second feature parity failed. Stop."
print("PASS: 24h same-currency amount matches frozen development features.")


Development rows compared: 3,731,672
Unmatched transaction IDs: 0
Feature mismatches (tolerance 1e-10): 0
Maximum absolute difference: 0.0
PASS: 24h same-currency amount matches frozen development features.


# Sender recency (strictly earlier timestamps)

In [20]:
def add_sender_recency(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add time since the sender's previous transaction.

    Only transactions at strictly earlier timestamps are used.
    Transactions occurring at the exact same timestamp do not
    count one another as previous activity.

    Sender identity:
        from_bank_id + from_account_id

    Model feature:
        log_sender_hours_since_prev_tx

    Encoding:
        -1 = no earlier sender transaction observed
        >= 0 = log1p(hours since previous transaction)
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "transaction_timestamp",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    sender_cols = [
        "from_bank_id",
        "from_account_id",
    ]

    # ---------------------------------------------------------
    # 1. Build one row per sender + unique timestamp.
    #
    # This prevents transactions occurring at the same exact
    # timestamp from treating one another as prior activity.
    # ---------------------------------------------------------

    sender_timestamps = (
        result[sender_cols + ["transaction_timestamp"]]
        .drop_duplicates()
        .sort_values(sender_cols + ["transaction_timestamp"])
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------
    # 2. Find the strictly previous timestamp for each sender.
    # ---------------------------------------------------------

    sender_timestamps["previous_sender_timestamp"] = sender_timestamps.groupby(
        sender_cols,
        sort=False,
    )["transaction_timestamp"].shift(1)

    # ---------------------------------------------------------
    # 3. Calculate hours since that previous transaction.
    # ---------------------------------------------------------

    sender_timestamps["sender_hours_since_prev_tx"] = (
        sender_timestamps["transaction_timestamp"]
        - sender_timestamps["previous_sender_timestamp"]
    ).dt.total_seconds() / 3600.0

    # ---------------------------------------------------------
    # 4. Log-transform recency.
    #
    # First observed sender transaction gets -1.
    # All actual recencies are >= 0.
    # ---------------------------------------------------------

    has_previous = sender_timestamps["previous_sender_timestamp"].notna()

    sender_timestamps["log_sender_hours_since_prev_tx"] = -1.0

    sender_timestamps.loc[
        has_previous,
        "log_sender_hours_since_prev_tx",
    ] = np.log1p(
        sender_timestamps.loc[
            has_previous,
            "sender_hours_since_prev_tx",
        ]
    )

    # ---------------------------------------------------------
    # 5. Merge recency back onto every original transaction.
    # ---------------------------------------------------------

    result = result.merge(
        sender_timestamps[
            sender_cols
            + [
                "transaction_timestamp",
                "sender_hours_since_prev_tx",
                "log_sender_hours_since_prev_tx",
            ]
        ],
        on=sender_cols + ["transaction_timestamp"],
        how="left",
        validate="many_to_one",
        sort=False,
    )

    result = (
        result.sort_values("_row_order")
        .drop(columns="_row_order")
        .reset_index(drop=True)
    )

    return result

In [21]:
assert "is_laundering" not in v3a3_combined_df.columns
v3b1_combined_df = add_sender_recency(v3a3_combined_df)
feature = "log_sender_hours_since_prev_tx"
assert len(v3b1_combined_df) == len(combined_df)
assert v3b1_combined_df["transaction_id"].is_unique
assert v3b1_combined_df[feature].notna().all()
assert (v3b1_combined_df[feature] >= -1).all()
print("Combined rows:", f"{len(v3b1_combined_df):,}")
print(
    "Rows with no previous sender transaction:",
    f"{(v3b1_combined_df[feature] == -1).sum():,}",
)
print(v3b1_combined_df[feature].describe(percentiles=[0.5, 0.9, 0.99]))


Combined rows: 5,078,345
Rows with no previous sender transaction: 502,628
count    5.078345e+06
mean     9.858943e-01
std      1.498593e+00
min     -1.000000e+00
50%      1.961149e-01
90%      3.214198e+00
99%      5.142735e+00
max      5.807542e+00
Name: log_sender_hours_since_prev_tx, dtype: float64


In [22]:
frozen_recency = pd.read_parquet(
    FROZEN_DEV_PATH,
    columns=["transaction_id", "log_sender_hours_since_prev_tx"],
)
recomputed_recency = v3b1_combined_df.loc[
    v3b1_combined_df["_source"].eq("development"),
    ["transaction_id", "log_sender_hours_since_prev_tx"],
]
assert frozen_recency["transaction_id"].is_unique
assert recomputed_recency["transaction_id"].is_unique
parity3 = recomputed_recency.merge(
    frozen_recency,
    on="transaction_id",
    how="outer",
    suffixes=("_recomputed", "_frozen"),
    indicator=True,
    validate="one_to_one",
)
unmatched = int(parity3["_merge"].ne("both").sum())
a = parity3["log_sender_hours_since_prev_tx_recomputed"].to_numpy()
b = parity3["log_sender_hours_since_prev_tx_frozen"].to_numpy()
close = np.isclose(a, b, rtol=1e-10, atol=1e-10, equal_nan=False)
mismatches = int((~close).sum())
max_abs_diff = float(np.max(np.abs(a - b))) if len(a) else 0.0
print("Development rows compared:", f"{len(recomputed_recency):,}")
print("Unmatched transaction IDs:", unmatched)
print("Feature mismatches (tolerance 1e-10):", mismatches)
print("Maximum absolute difference:", max_abs_diff)
assert unmatched == 0, "Development transaction ID parity failed. Stop."
assert mismatches == 0, "Sender recency parity failed. Stop."
print("PASS: sender recency matches frozen V3-E1 development values.")


Development rows compared: 3,731,672
Unmatched transaction IDs: 0
Feature mismatches (tolerance 1e-10): 0
Maximum absolute difference: 0.0
PASS: sender recency matches frozen V3-E1 development values.


# Sender-relative amount (historical same-currency median)

In [23]:
def add_sender_relative_amount(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add the current amount paid relative to the sender's
    historical median amount paid in the same payment currency.

    Historical information uses only transactions occurring at
    strictly earlier timestamps.

    Sender identity:
        from_bank_id + from_account_id

    Historical comparison:
        same payment_currency

    Model feature:
        log_amount_paid_vs_sender_median
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "transaction_timestamp",
        "payment_currency",
        "amount_paid",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    # Avoid grouping/merge ambiguity for missing currencies.
    result["_payment_currency_key"] = (
        result["payment_currency"].astype("string").fillna("__MISSING__")
    )

    group_cols = [
        "from_bank_id",
        "from_account_id",
        "_payment_currency_key",
    ]

    sort_cols = group_cols + ["transaction_timestamp"]

    # Stable deterministic ordering.
    if "transaction_id" in result.columns:
        sort_cols = sort_cols + ["transaction_id"]

    result = result.sort_values(
        sort_cols,
        kind="mergesort",
    ).reset_index(drop=True)

    # ---------------------------------------------------------
    # 1. Shift by one transaction.
    #
    # For the FIRST transaction at any timestamp, this creates
    # a history containing only earlier rows.
    # ---------------------------------------------------------

    result["_previous_amount"] = result.groupby(
        group_cols,
        sort=False,
        dropna=False,
    )["amount_paid"].shift(1)

    # ---------------------------------------------------------
    # 2. Expanding median of previous amounts.
    #
    # At the first row of each timestamp this represents the
    # exact median of all strictly earlier transactions.
    # ---------------------------------------------------------

    result["_preliminary_historical_median"] = (
        result.groupby(
            group_cols,
            sort=False,
            dropna=False,
        )["_previous_amount"]
        .expanding()
        .median()
        .reset_index(
            level=list(range(len(group_cols))),
            drop=True,
        )
    )

    timestamp_cols = group_cols + ["transaction_timestamp"]

    # ---------------------------------------------------------
    # 3. Keep ONLY the first row at each exact timestamp.
    #
    # Why?
    #
    # Later rows at the same timestamp would otherwise start
    # seeing other transactions at that timestamp.
    #
    # The first row's historical median is guaranteed to use
    # strictly earlier timestamps only.
    # ---------------------------------------------------------

    timestamp_history = (
        result.groupby(
            timestamp_cols,
            sort=False,
            dropna=False,
        )
        .head(1)[timestamp_cols + ["_preliminary_historical_median"]]
        .rename(
            columns={
                "_preliminary_historical_median": "sender_historical_median_amount_paid"
            }
        )
    )

    # Drop preliminary version before merging the safe value.
    result = result.drop(
        columns=[
            "_previous_amount",
            "_preliminary_historical_median",
        ]
    )

    result = result.merge(
        timestamp_history,
        on=timestamp_cols,
        how="left",
        validate="many_to_one",
        sort=False,
    )

    # ---------------------------------------------------------
    # 4. Create the actual model feature.
    #
    # NaN remains NaN when there is no previous history.
    # The pipeline's SimpleImputer will learn how to impute
    # those values inside each training fold.
    # ---------------------------------------------------------

    result["log_amount_paid_vs_sender_median"] = np.log1p(
        result["amount_paid"]
    ) - np.log1p(result["sender_historical_median_amount_paid"])

    # Restore original row order.
    result = (
        result.sort_values("_row_order")
        .drop(
            columns=[
                "_row_order",
                "_payment_currency_key",
            ]
        )
        .reset_index(drop=True)
    )

    return result

In [24]:
assert "is_laundering" not in v3b1_combined_df.columns
v3c1_combined_df = add_sender_relative_amount(v3b1_combined_df)
feature = "log_amount_paid_vs_sender_median"

assert len(v3c1_combined_df) == len(combined_df)
assert v3c1_combined_df["transaction_id"].is_unique

print("Combined rows:", f"{len(v3c1_combined_df):,}")
print("Missing relative-amount values:", f"{v3c1_combined_df[feature].isna().sum():,}")
print(v3c1_combined_df[feature].describe(percentiles=[0.5, 0.9, 0.99]))

Combined rows: 5,078,345
Missing relative-amount values: 510,347
count    4.567998e+06
mean    -8.129783e-02
std      2.667595e+00
min     -1.973844e+01
50%      0.000000e+00
90%      2.783369e+00
99%      8.186102e+00
max      2.108953e+01
Name: log_amount_paid_vs_sender_median, dtype: float64


In [25]:
frozen_relative = pd.read_parquet(
    FROZEN_DEV_PATH,
    columns=["transaction_id", "log_amount_paid_vs_sender_median"],
)
recomputed_relative = v3c1_combined_df.loc[
    v3c1_combined_df["_source"].eq("development"),
    ["transaction_id", "log_amount_paid_vs_sender_median"],
]

assert frozen_relative["transaction_id"].is_unique
assert recomputed_relative["transaction_id"].is_unique

parity4 = recomputed_relative.merge(
    frozen_relative,
    on="transaction_id",
    how="outer",
    suffixes=("_recomputed", "_frozen"),
    indicator=True,
    validate="one_to_one",
)

unmatched = int(parity4["_merge"].ne("both").sum())
a = parity4["log_amount_paid_vs_sender_median_recomputed"].to_numpy(dtype=float)
b = parity4["log_amount_paid_vs_sender_median_frozen"].to_numpy(dtype=float)
missing_pattern_mismatches = int(np.count_nonzero(np.isnan(a) != np.isnan(b)))
close = np.isclose(a, b, rtol=1e-10, atol=1e-10, equal_nan=True)
mismatches = int(np.count_nonzero(~close))
finite_pairs = np.isfinite(a) & np.isfinite(b)
max_abs_diff = (
    float(np.max(np.abs(a[finite_pairs] - b[finite_pairs])))
    if finite_pairs.any()
    else 0.0
)

print("Development rows compared:", f"{len(recomputed_relative):,}")
print("Unmatched transaction IDs:", unmatched)
print("Missing-value pattern mismatches:", missing_pattern_mismatches)
print("Feature mismatches (tolerance 1e-10):", mismatches)
print("Maximum finite absolute difference:", max_abs_diff)
assert len(recomputed_relative) == 3_731_672
assert unmatched == 0, "Transaction ID mismatch; stop."
assert missing_pattern_mismatches == 0, "Missing-value history mismatch; stop."
assert mismatches == 0, "Sender-relative amount parity failed; stop."
print("PASS: sender-relative amount matches frozen V3-E1 development values.")

Development rows compared: 3,731,672
Unmatched transaction IDs: 0
Missing-value pattern mismatches: 0
Feature mismatches (tolerance 1e-10): 0
Maximum finite absolute difference: 0.0
PASS: sender-relative amount matches frozen V3-E1 development values.


# Previously seen receiver (strictly earlier relationship history)

In [26]:
def add_receiver_seen_before(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Indicate whether the sender has previously transacted
    with the same receiver.

    Only strictly earlier timestamps count as history.

    Sender:
        from_bank_id + from_account_id

    Receiver:
        to_bank_id + to_account_id

    Feature:
        receiver_seen_before

    Values:
        0 = no earlier sender-receiver transaction observed
        1 = sender has previously transacted with receiver
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "to_bank_id",
        "to_account_id",
        "transaction_timestamp",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    relationship_cols = [
        "from_bank_id",
        "from_account_id",
        "to_bank_id",
        "to_account_id",
    ]

    # ---------------------------------------------------------
    # One row per sender-receiver relationship + timestamp.
    #
    # This makes same-timestamp transactions share the same
    # historical state.
    # ---------------------------------------------------------

    relationship_timestamps = (
        result[relationship_cols + ["transaction_timestamp"]]
        .drop_duplicates()
        .sort_values(relationship_cols + ["transaction_timestamp"])
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------
    # cumcount = 0 for first observed timestamp
    #            1+ for later timestamps.
    #
    # Therefore later relationship timestamps mean this
    # receiver was already known to this sender.
    # ---------------------------------------------------------

    relationship_timestamps["receiver_seen_before"] = (
        relationship_timestamps.groupby(
            relationship_cols,
            sort=False,
            dropna=False,
        )
        .cumcount()
        .gt(0)
        .astype("int8")
    )

    # ---------------------------------------------------------
    # Merge back to the original transactions.
    # ---------------------------------------------------------

    result = result.merge(
        relationship_timestamps[
            relationship_cols
            + [
                "transaction_timestamp",
                "receiver_seen_before",
            ]
        ],
        on=(relationship_cols + ["transaction_timestamp"]),
        how="left",
        validate="many_to_one",
        sort=False,
    )

    result = (
        result.sort_values("_row_order")
        .drop(columns="_row_order")
        .reset_index(drop=True)
    )

    return result

In [27]:
assert "is_laundering" not in v3c1_combined_df.columns
v3d1_combined_df = add_receiver_seen_before(v3c1_combined_df)
feature = "receiver_seen_before"
assert len(v3d1_combined_df) == len(v3c1_combined_df)
assert v3d1_combined_df["transaction_id"].equals(v3c1_combined_df["transaction_id"])
assert v3d1_combined_df[feature].notna().all()
assert set(v3d1_combined_df[feature].unique()).issubset({0, 1})
print("Feature:", feature)
print("Combined rows:", f"{len(v3d1_combined_df):,}")
print("Observed feature counts:")
print(v3d1_combined_df[feature].value_counts(dropna=False).sort_index())

Feature: receiver_seen_before
Combined rows: 5,078,345
Observed feature counts:
receiver_seen_before
0    1028214
1    4050131
Name: count, dtype: int64


In [28]:
frozen_seen = pd.read_parquet(
    FROZEN_DEV_PATH,
    columns=["transaction_id", "receiver_seen_before"],
)
recomputed_seen = v3d1_combined_df.loc[
    v3d1_combined_df["_source"].eq("development"),
    ["transaction_id", "receiver_seen_before"],
].copy()

assert frozen_seen["transaction_id"].is_unique
assert recomputed_seen["transaction_id"].is_unique

matched_seen = frozen_seen.merge(
    recomputed_seen,
    on="transaction_id",
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("_frozen", "_recomputed"),
)
unmatched_ids = int(matched_seen["_merge"].ne("both").sum())
mismatches = int(
    (
        matched_seen["receiver_seen_before_frozen"]
        != matched_seen["receiver_seen_before_recomputed"]
    ).sum()
)

print("Development rows compared:", f"{len(matched_seen):,}")
print("Unmatched transaction IDs:", unmatched_ids)
print("Feature mismatches:", mismatches)

assert len(matched_seen) == 3_731_672
assert unmatched_ids == 0
assert mismatches == 0
print("PASS: receiver_seen_before matches frozen V3-E1 development values.")

Development rows compared: 3,731,672
Unmatched transaction IDs: 0
Feature mismatches: 0
PASS: receiver_seen_before matches frozen V3-E1 development values.


# Sender–receiver prior transaction count

In [29]:
def add_sender_receiver_prior_tx_count(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add the number of strictly earlier transactions between
    the same sender and receiver.

    Sender identity:
        from_bank_id + from_account_id

    Receiver identity:
        to_bank_id + to_account_id

    Only transactions with timestamp < current timestamp
    contribute to the count.

    Transactions occurring at the same exact timestamp do
    not count one another.
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "to_bank_id",
        "to_account_id",
        "transaction_timestamp",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    relationship_cols = [
        "from_bank_id",
        "from_account_id",
        "to_bank_id",
        "to_account_id",
    ]

    # ---------------------------------------------------------
    # 1. Count how many transactions occurred for each
    #    sender-receiver pair at each exact timestamp.
    # ---------------------------------------------------------

    timestamp_counts = (
        result.groupby(
            relationship_cols + ["transaction_timestamp"],
            dropna=False,
        )
        .size()
        .rename("tx_at_timestamp")
        .reset_index()
    )

    timestamp_counts = timestamp_counts.sort_values(
        relationship_cols + ["transaction_timestamp"]
    ).reset_index(drop=True)

    # ---------------------------------------------------------
    # 2. Cumulative number of relationship transactions.
    #
    # Shift first so the current timestamp is excluded.
    # ---------------------------------------------------------

    timestamp_counts["sender_receiver_prior_tx_count"] = (
        timestamp_counts.groupby(
            relationship_cols,
            sort=False,
            dropna=False,
        )["tx_at_timestamp"]
        .transform(lambda x: x.cumsum().shift(1, fill_value=0))
        .astype("int32")
    )

    # ---------------------------------------------------------
    # 3. Merge the past-only count back onto every transaction.
    # ---------------------------------------------------------

    result = result.merge(
        timestamp_counts[
            relationship_cols
            + [
                "transaction_timestamp",
                "sender_receiver_prior_tx_count",
            ]
        ],
        on=(relationship_cols + ["transaction_timestamp"]),
        how="left",
        validate="many_to_one",
        sort=False,
    )

    result = (
        result.sort_values("_row_order")
        .drop(columns="_row_order")
        .reset_index(drop=True)
    )

    return result

In [30]:
assert "is_laundering" not in v3d1_combined_df.columns
v3d2_combined_df = add_sender_receiver_prior_tx_count(v3d1_combined_df)
feature = "sender_receiver_prior_tx_count"

assert len(v3d2_combined_df) == len(v3d1_combined_df)
assert v3d2_combined_df["transaction_id"].equals(v3d1_combined_df["transaction_id"])
assert v3d2_combined_df[feature].notna().all()
assert (v3d2_combined_df[feature] >= 0).all()

print("Feature:", feature)
print("Combined transactions:", f"{len(v3d2_combined_df):,}")
print("Feature summary:")
print(v3d2_combined_df[feature].describe())

Feature: sender_receiver_prior_tx_count
Combined transactions: 5,078,345
Feature summary:
count    5.078345e+06
mean     7.444157e+00
std      7.816555e+00
min      0.000000e+00
25%      1.000000e+00
50%      5.000000e+00
75%      1.200000e+01
max      8.800000e+01
Name: sender_receiver_prior_tx_count, dtype: float64


In [31]:
frozen_prior = pd.read_parquet(
    FROZEN_DEV_PATH,
    columns=["transaction_id", "sender_receiver_prior_tx_count"],
)
recomputed_prior = v3d2_combined_df.loc[
    v3d2_combined_df["_source"].eq("development"),
    ["transaction_id", "sender_receiver_prior_tx_count"],
].copy()

assert frozen_prior["transaction_id"].is_unique
assert recomputed_prior["transaction_id"].is_unique

matched_prior = frozen_prior.merge(
    recomputed_prior,
    on="transaction_id",
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("_frozen", "_recomputed"),
)
unmatched_ids = int(matched_prior["_merge"].ne("both").sum())
mismatches = int(
    (
        matched_prior["sender_receiver_prior_tx_count_frozen"]
        != matched_prior["sender_receiver_prior_tx_count_recomputed"]
    ).sum()
)

print("Development rows compared:", f"{len(matched_prior):,}")
print("Unmatched transaction IDs:", unmatched_ids)
print("Feature mismatches:", mismatches)

assert len(matched_prior) == 3_731_672
assert unmatched_ids == 0
assert mismatches == 0
print("PASS: sender_receiver_prior_tx_count matches frozen V3-E1 development values.")

Development rows compared: 3,731,672
Unmatched transaction IDs: 0
Feature mismatches: 0
PASS: sender_receiver_prior_tx_count matches frozen V3-E1 development values.


# Distinct receivers in the previous 24 hours

In [32]:
from collections import Counter, deque


def add_sender_distinct_receivers_24h(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add the number of distinct receivers used by the sender
    during the previous 24 hours.

    Window:
        [t - 24 hours, t)

    Therefore:
    - current timestamp is excluded
    - same-timestamp transactions cannot see one another
    - future transactions are excluded

    Sender:
        from_bank_id + from_account_id

    Receiver:
        to_bank_id + to_account_id
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "to_bank_id",
        "to_account_id",
        "transaction_timestamp",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    sender_cols = [
        "from_bank_id",
        "from_account_id",
    ]

    receiver_cols = [
        "to_bank_id",
        "to_account_id",
    ]

    # ---------------------------------------------------------
    # 1. One event per sender-receiver-timestamp.
    #
    # If the same sender sends multiple transactions to the
    # same receiver at exactly the same timestamp, that
    # receiver should count only once for distinct-receiver
    # history.
    # ---------------------------------------------------------

    events = (
        result[sender_cols + receiver_cols + ["transaction_timestamp"]]
        .drop_duplicates()
        .sort_values(
            sender_cols + ["transaction_timestamp"] + receiver_cols,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    historical_counts = []

    window_size = pd.Timedelta(hours=24)

    # ---------------------------------------------------------
    # 2. Process each sender chronologically.
    # ---------------------------------------------------------

    for sender_key, sender_events in events.groupby(
        sender_cols,
        sort=False,
        dropna=False,
    ):
        # Number of currently active timestamp-events for
        # each receiver inside the 24h window.
        active_receiver_counts = Counter()

        # Historical receiver events currently inside window.
        # Each element:
        #     (timestamp, receiver_key)
        active_window = deque()

        sender_key = sender_key if isinstance(sender_key, tuple) else (sender_key,)

        current_timestamp = None
        receivers_at_current_timestamp = []

        def process_timestamp(
            timestamp,
            receivers,
        ):
            """
            Calculate history BEFORE adding receivers from the
            current timestamp.
            """

            cutoff = timestamp - window_size

            # Remove events older than t - 24h.
            #
            # Notice '<', not '<='.
            #
            # An event exactly 24h earlier remains inside
            # [t - 24h, t).
            while active_window and active_window[0][0] < cutoff:
                old_timestamp, old_receiver = active_window.popleft()

                active_receiver_counts[old_receiver] -= 1

                if active_receiver_counts[old_receiver] == 0:
                    del active_receiver_counts[old_receiver]

            # History is recorded BEFORE current timestamp
            # receivers are added.
            historical_counts.append(
                (
                    *sender_key,
                    timestamp,
                    len(active_receiver_counts),
                )
            )

            # Now current timestamp becomes available to
            # FUTURE transactions.
            for receiver in receivers:
                active_window.append((timestamp, receiver))

                active_receiver_counts[receiver] += 1

        # -----------------------------------------------------
        # Walk through sender events, one timestamp block
        # at a time.
        # -----------------------------------------------------

        for row in sender_events.itertuples(
            index=False,
            name=None,
        ):
            # Column order in events:
            # sender cols,
            # receiver cols,
            # timestamp

            receiver = (
                row[2],
                row[3],
            )

            timestamp = row[4]

            if current_timestamp is None:
                current_timestamp = timestamp

            # We reached a new timestamp.
            if timestamp != current_timestamp:
                process_timestamp(
                    current_timestamp,
                    receivers_at_current_timestamp,
                )

                current_timestamp = timestamp
                receivers_at_current_timestamp = []

            receivers_at_current_timestamp.append(receiver)

        # Process final timestamp for this sender.
        if current_timestamp is not None:
            process_timestamp(
                current_timestamp,
                receivers_at_current_timestamp,
            )

    # ---------------------------------------------------------
    # 3. Build sender/timestamp lookup table.
    # ---------------------------------------------------------

    history_df = pd.DataFrame(
        historical_counts,
        columns=[
            "from_bank_id",
            "from_account_id",
            "transaction_timestamp",
            "sender_distinct_receivers_24h",
        ],
    )

    history_df["sender_distinct_receivers_24h"] = history_df[
        "sender_distinct_receivers_24h"
    ].astype("int32")

    # ---------------------------------------------------------
    # 4. Merge history back to every original transaction.
    # ---------------------------------------------------------

    result = result.merge(
        history_df,
        on=(sender_cols + ["transaction_timestamp"]),
        how="left",
        validate="many_to_one",
        sort=False,
    )

    result = (
        result.sort_values("_row_order")
        .drop(columns="_row_order")
        .reset_index(drop=True)
    )

    return result

In [33]:
assert "is_laundering" not in v3d2_combined_df.columns
v3e1_combined_df = add_sender_distinct_receivers_24h(v3d2_combined_df)
feature = "sender_distinct_receivers_24h"

assert len(v3e1_combined_df) == len(v3d2_combined_df)
assert v3e1_combined_df["transaction_id"].equals(v3d2_combined_df["transaction_id"])
assert v3e1_combined_df[feature].notna().all()
assert (v3e1_combined_df[feature] >= 0).all()

print("Feature:", feature)
print("Combined transactions:", f"{len(v3e1_combined_df):,}")
print("Feature summary:")
print(v3e1_combined_df[feature].describe())

Feature: sender_distinct_receivers_24h
Combined transactions: 5,078,345
Feature summary:
count    5.078345e+06
mean     3.654018e+02
std      1.538572e+03
min      0.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      1.253800e+04
Name: sender_distinct_receivers_24h, dtype: float64


In [34]:
frozen_distinct = pd.read_parquet(
    FROZEN_DEV_PATH,
    columns=["transaction_id", "sender_distinct_receivers_24h"],
)
recomputed_distinct = v3e1_combined_df.loc[
    v3e1_combined_df["_source"].eq("development"),
    ["transaction_id", "sender_distinct_receivers_24h"],
].copy()

assert frozen_distinct["transaction_id"].is_unique
assert recomputed_distinct["transaction_id"].is_unique

matched_distinct = frozen_distinct.merge(
    recomputed_distinct,
    on="transaction_id",
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("_frozen", "_recomputed"),
)
unmatched_ids = int(matched_distinct["_merge"].ne("both").sum())
mismatches = int(
    (
        matched_distinct["sender_distinct_receivers_24h_frozen"]
        != matched_distinct["sender_distinct_receivers_24h_recomputed"]
    ).sum()
)

print("Development rows compared:", f"{len(matched_distinct):,}")
print("Unmatched transaction IDs:", unmatched_ids)
print("Feature mismatches:", mismatches)

assert len(matched_distinct) == 3_731_672
assert unmatched_ids == 0
assert mismatches == 0
print("PASS: sender_distinct_receivers_24h matches frozen V3-E1 development values.")

Development rows compared: 3,731,672
Unmatched transaction IDs: 0
Feature mismatches: 0
PASS: sender_distinct_receivers_24h matches frozen V3-E1 development values.


# Audit

In [35]:
# 9.1 Define the exact frozen model input schema
NUMERIC_FEATURES = [
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "same_currency_flag",
    "same_bank_flag",
    "log_amount_received",
    "log_amount_paid",
    "sender_tx_count_1h",
    "log_sender_amount_paid_sum_24h_same_currency",
    "log_sender_hours_since_prev_tx",
    "log_amount_paid_vs_sender_median",
    "receiver_seen_before",
    "sender_receiver_prior_tx_count",
    "sender_distinct_receivers_24h",
]
CATEGORICAL_FEATURES = [
    "receiving_currency",
    "payment_currency",
    "payment_format",
]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

assert len(FEATURE_COLUMNS) == 17
assert len(v3e1_combined_df) == 5_078_345
assert v3e1_combined_df["transaction_id"].is_unique
assert "is_laundering" not in v3e1_combined_df.columns
assert set(FEATURE_COLUMNS).issubset(v3e1_combined_df.columns)

# Read frozen reference columns only, reducing memory use.
frozen_all = pd.read_parquet(
    FROZEN_DEV_PATH, columns=["transaction_id", *FEATURE_COLUMNS]
)
recomputed_dev = v3e1_combined_df.loc[
    v3e1_combined_df["_source"].eq("development"),
    ["transaction_id", *FEATURE_COLUMNS],
]

assert len(frozen_all) == len(recomputed_dev) == 3_731_672
assert frozen_all["transaction_id"].is_unique
assert recomputed_dev["transaction_id"].is_unique

# The same ID set is necessary before feature-by-feature comparison.
reference_ids = pd.Index(frozen_all["transaction_id"])
recomputed_by_id = recomputed_dev.set_index("transaction_id")
assert reference_ids.isin(recomputed_by_id.index).all(), "Development IDs missing"
assert recomputed_by_id.index.isin(reference_ids).all(), "Unexpected development IDs"

print("Feature columns:", len(FEATURE_COLUMNS))
print("Development IDs matched:", f"{len(reference_ids):,}")
print("Target in feature frame:", "is_laundering" in v3e1_combined_df.columns)

Feature columns: 17
Development IDs matched: 3,731,672
Target in feature frame: False


In [36]:
# 9.2 Full development parity: numeric values, NaNs and categoricals
# Align by ID, not row position; do not alter the feature values.
parity_summary = []
for feature in FEATURE_COLUMNS:
    reference = frozen_all[feature].reset_index(drop=True)
    computed = recomputed_by_id[feature].reindex(reference_ids).reset_index(drop=True)

    if feature in NUMERIC_FEATURES:
        a = pd.to_numeric(reference, errors="raise").to_numpy(dtype="float64")
        b = pd.to_numeric(computed, errors="raise").to_numpy(dtype="float64")
        nan_mismatch = int(np.count_nonzero(np.isnan(a) != np.isnan(b)))
        finite_a = np.isfinite(a)
        finite_b = np.isfinite(b)
        finite_mismatch = int(np.count_nonzero(finite_a != finite_b))
        value_mismatch = int(
            np.count_nonzero(~np.isclose(a, b, rtol=0, atol=1e-10, equal_nan=True))
        )
        mismatches = max(nan_mismatch, finite_mismatch, value_mismatch)
    else:
        a = reference.astype("string").fillna("__MISSING__")
        b = computed.astype("string").fillna("__MISSING__")
        mismatches = int(a.ne(b).sum())

    parity_summary.append({"feature": feature, "mismatches": mismatches})
    print(f"{feature}: {mismatches:,} mismatches")

parity_df = pd.DataFrame(parity_summary)
assert (parity_df["mismatches"] == 0).all(), (
    "Full parity check failed; do not export test features."
)
print("PASS: All 17 features match the frozen development data.")

hour_of_day: 0 mismatches
day_of_week: 0 mismatches
is_weekend: 0 mismatches
same_currency_flag: 0 mismatches
same_bank_flag: 0 mismatches
log_amount_received: 0 mismatches
log_amount_paid: 0 mismatches
sender_tx_count_1h: 0 mismatches
log_sender_amount_paid_sum_24h_same_currency: 0 mismatches
log_sender_hours_since_prev_tx: 0 mismatches
log_amount_paid_vs_sender_median: 0 mismatches
receiver_seen_before: 0 mismatches
sender_receiver_prior_tx_count: 0 mismatches
sender_distinct_receivers_24h: 0 mismatches
receiving_currency: 0 mismatches
payment_currency: 0 mismatches
payment_format: 0 mismatches
PASS: All 17 features match the frozen development data.


In [37]:
# 9.3 Extract and audit only the holdout rows
holdout_features = v3e1_combined_df.loc[
    v3e1_combined_df["_source"].eq("holdout"),
    ["transaction_id", "transaction_timestamp", *FEATURE_COLUMNS],
].copy()

assert len(holdout_features) == len(test_source) == 1_346_673
assert holdout_features["transaction_id"].is_unique
assert holdout_features["transaction_id"].notna().all()
assert holdout_features["transaction_timestamp"].notna().all()
assert (holdout_features["transaction_timestamp"] >= TEST_START).all()
assert holdout_features["transaction_timestamp"].is_monotonic_increasing
assert set(holdout_features["transaction_id"]) == set(test_source["transaction_id"])

for feature in NUMERIC_FEATURES:
    values = pd.to_numeric(holdout_features[feature], errors="raise")
    assert not np.isinf(values.to_numpy(dtype="float64")).any(), (
        f"Infinite values in {feature}"
    )

missing = holdout_features[FEATURE_COLUMNS].isna().sum()
print("Holdout transactions:", f"{len(holdout_features):,}")
print(
    "Holdout date range:",
    holdout_features["transaction_timestamp"].min(),
    "to",
    holdout_features["transaction_timestamp"].max(),
)
print("Missing model-input values (imputed later by saved pipeline):")
print(missing[missing.gt(0)].to_string() if missing.gt(0).any() else "None")
print("PASS: Holdout identity, chronology, and numeric validity.")

Holdout transactions: 1,346,673
Holdout date range: 2022-09-08 00:00:00 to 2022-09-18 16:18:00
Missing model-input values (imputed later by saved pipeline):
log_amount_paid_vs_sender_median    4112
PASS: Holdout identity, chronology, and numeric validity.


In [38]:
# 9.4 Attach untouched labels by ID *after* feature construction.
# Do not print label distributions or evaluate the model here.
TARGET = "is_laundering"
assert TARGET in test_source.columns
labels = test_source[["transaction_id", TARGET]].copy()
assert labels["transaction_id"].is_unique
assert labels[TARGET].notna().all()
assert labels[TARGET].isin([0, 1]).all()

holdout_export = holdout_features.merge(
    labels,
    on="transaction_id",
    how="left",
    validate="one_to_one",
    indicator=True,
    sort=False,
)
assert holdout_export["_merge"].eq("both").all()
holdout_export = holdout_export.drop(columns="_merge")
assert len(holdout_export) == 1_346_673
assert holdout_export["transaction_timestamp"].is_monotonic_increasing
assert holdout_export.columns.tolist() == [
    "transaction_id",
    "transaction_timestamp",
    *FEATURE_COLUMNS,
    TARGET,
]

# Explicit non-overwrite policy: never silently replace the first prepared holdout.
OUTPUT_PATH = PROCESSED_DIR / "v3e1_test_sep8_plus.parquet"
assert not OUTPUT_PATH.exists(), (
    f"Output already exists: {OUTPUT_PATH}. Review it before replacing anything."
)
holdout_export.to_parquet(OUTPUT_PATH, index=False)

# Verify the file can be reopened with the expected structure.
read_back = pd.read_parquet(
    OUTPUT_PATH, columns=["transaction_id", "transaction_timestamp", *FEATURE_COLUMNS]
)
assert len(read_back) == len(holdout_export)
assert read_back["transaction_id"].equals(holdout_export["transaction_id"])
assert read_back["transaction_timestamp"].equals(
    holdout_export["transaction_timestamp"]
)
assert read_back[FEATURE_COLUMNS].columns.tolist() == FEATURE_COLUMNS

print("Saved:", OUTPUT_PATH)
print("Rows:", f"{len(holdout_export):,}")
print("Feature count:", len(FEATURE_COLUMNS))
print("Target labels attached only after features generated: True")
print("Final test model predictions: NOT computed")
print("PASS: V3-E1 holdout feature dataset prepared.")

Saved: c:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\processed\v3e1_test_sep8_plus.parquet
Rows: 1,346,673
Feature count: 17
Target labels attached only after features generated: True
Final test model predictions: NOT computed
PASS: V3-E1 holdout feature dataset prepared.
